In [16]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_score

In [35]:
# setup and feature engineering
df = pd.read_csv('../../data/processed/neighborhood_equity_model.csv')

# set neighborhood as index so removed from feature columns
df.set_index("neighborhood", inplace=True)

# select feature columns and scale
X = df.drop(columns=["avgOverallEquityScore"])

# target column - unscaled
target_col = 'avgOverallEquityScore'
y = df[target_col].round().astype(int).values

# only one instance with class 5 so merge with class 4:
y[y == 5] = 4

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Logist Regression Models

These models will use both the train/test split data, as well as the complete data set with cross_val_score to evaluate.

### Basic Logistic Regression without Regularization

In [36]:
# with trian_test_split
log_reg = LogisticRegression(max_iter=1000, penalty=None)

log_reg.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = log_reg.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# coefficients
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': log_reg.coef_[0]
})
print(coef_df.sort_values(by='coefficient', key=abs, ascending=False))

Confusion Matrix:
[[9 0 0]
 [1 1 0]
 [0 1 4]]

Classification Report:
              precision    recall  f1-score   support

           2       0.90      1.00      0.95         9
           3       0.50      0.50      0.50         2
           4       1.00      0.80      0.89         5

    accuracy                           0.88        16
   macro avg       0.80      0.77      0.78        16
weighted avg       0.88      0.88      0.87        16

                              feature  coefficient
8             Median Household Income   -33.205688
4                       Percent Asian   -30.116525
11          Percent Living in Poverty    27.336072
14                    percent_less_hs    24.601571
10                  Median Home Value    17.175073
7           Percent Two or More Races   -15.517591
18              program_count_per1000   -15.093767
17                  percent_bachelors   -14.054673
9                   Per Capita Income   -14.010450
0                    Percent Hispanic  

In [37]:
# with cross_val_score
log_reg_cv = LogisticRegression(max_iter=1000, penalty=None)
scores = cross_val_score(log_reg_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.68080808 0.6048951  0.66550117 0.92207792 0.77777778]
Mean F1: 0.7302120102120101


### Logistic Regression with L1 Regularization

In [38]:
lasso = LogisticRegression(max_iter=10000, penalty='l1', C=0.5, solver='saga')

lasso.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = lasso.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# coefficients
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lasso.coef_[0]
})
print(coef_df.sort_values(by='coefficient', key=abs, ascending=False))

Confusion Matrix:
[[8 1 0]
 [1 1 0]
 [0 1 4]]

Classification Report:
              precision    recall  f1-score   support

           2       0.89      0.89      0.89         9
           3       0.33      0.50      0.40         2
           4       1.00      0.80      0.89         5

    accuracy                           0.81        16
   macro avg       0.74      0.73      0.73        16
weighted avg       0.85      0.81      0.83        16

                              feature  coefficient
11          Percent Living in Poverty     0.835015
14                    percent_less_hs     0.781860
15                      percent_at_hs     0.516492
8             Median Household Income    -0.401686
5   Percent Hawaiian/Pacific Islander     0.140914
17                  percent_bachelors    -0.081805
0                    Percent Hispanic     0.066480
7           Percent Two or More Races    -0.042640
12        Percent Families in Poverty     0.000000
18              program_count_per1000  

In [39]:
# with cross_val_score
lasso_cv = LogisticRegression(max_iter=10000, penalty='l1', C=0.5, solver='saga')
scores = cross_val_score(lasso_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.72466422 0.72610723 0.52777778 0.92207792 0.54047619]
Mean F1: 0.6882206682206682


### Logistic Regression with L2 Regularization

In [40]:
ridge = LogisticRegression(max_iter=1000, penalty='l2', C=0.5)

ridge.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = ridge.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# coefficients
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': ridge.coef_[0]
})
print(coef_df.sort_values(by='coefficient', key=abs, ascending=False))

Confusion Matrix:
[[9 0 0]
 [1 1 0]
 [0 1 4]]

Classification Report:
              precision    recall  f1-score   support

           2       0.90      1.00      0.95         9
           3       0.50      0.50      0.50         2
           4       1.00      0.80      0.89         5

    accuracy                           0.88        16
   macro avg       0.80      0.77      0.78        16
weighted avg       0.88      0.88      0.87        16

                              feature  coefficient
11          Percent Living in Poverty     0.624658
8             Median Household Income    -0.553682
14                    percent_less_hs     0.548094
4                       Percent Asian    -0.470164
12        Percent Families in Poverty     0.453307
17                  percent_bachelors    -0.413279
0                    Percent Hispanic     0.389501
18              program_count_per1000    -0.310512
9                   Per Capita Income    -0.309869
15                      percent_at_hs  

In [42]:
# with cross_val_score
ridge_cv = LogisticRegression(max_iter=1000, penalty='l2', C=0.5)
scores = cross_val_score(ridge_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.62942613 0.58333333 0.57387057 1.         0.70238095]
Mean F1: 0.6978021978021979


### Logistic Regression with Elasticnet Regularization

In [43]:
elastic = LogisticRegression(max_iter=1000, penalty='elasticnet', l1_ratio=0.5, solver='saga')

elastic.fit(X_train_scaled, y_train)

# predict on the test set
y_pred = elastic.predict(X_test_scaled)

# evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# coefficients
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': elastic.coef_[0]
})
print(coef_df.sort_values(by='coefficient', key=abs, ascending=False))

Confusion Matrix:
[[9 0 0]
 [1 1 0]
 [0 1 4]]

Classification Report:
              precision    recall  f1-score   support

           2       0.90      1.00      0.95         9
           3       0.50      0.50      0.50         2
           4       1.00      0.80      0.89         5

    accuracy                           0.88        16
   macro avg       0.80      0.77      0.78        16
weighted avg       0.88      0.88      0.87        16

                              feature  coefficient
11          Percent Living in Poverty     0.903611
8             Median Household Income    -0.880598
14                    percent_less_hs     0.705500
17                  percent_bachelors    -0.452237
4                       Percent Asian    -0.397552
12        Percent Families in Poverty     0.371400
18              program_count_per1000    -0.359043
0                    Percent Hispanic     0.301177
7           Percent Two or More Races    -0.286473
9                   Per Capita Income  

In [44]:
# with cross_val_score
elastic_cv = LogisticRegression(max_iter=1000, penalty='elasticnet', l1_ratio=0.5, solver='saga')
scores = cross_val_score(elastic_cv, X_scaled, y, cv=5, scoring='f1_macro')
print("F1 scores:", scores)
print("Mean F1:", scores.mean())

F1 scores: [0.72610723 0.58333333 0.57387057 1.         0.77991453]
Mean F1: 0.7326451326451326
